# Introducción al Streaming y Flujos de Eventos

---

## ¿Qué es el Data Streaming?

El Data Streaming es el paradigma de procesar datos en tiempo real, gestionando flujos de datos no acotados (unbounded data).  
A diferencia de los conjuntos de datos tradicionales, un flujo es una secuencia infinita de mensajes generados gradualmente a lo largo del tiempo.

### ¿Qué es un Evento?

Un evento es la unidad básica de un flujo: un objeto pequeño, inmutable y autodescriptivo que registra que algo sucedió en un momento específico.  
Ejemplos:
- Acciones de usuario: clics, compras, vistas de página.
- Lecturas de sensores: temperatura, utilización de CPU.

---

## Batch vs. Streaming: El Cambio de Paradigma

| Batch | Streaming |
|-------|-----------|
| Procesa grandes volúmenes de datos acumulados en intervalos fijos. | Procesa datos en tiempo real, con baja latencia. |
| Latencia alta, permite análisis complejos. | Ideal para responder a eventos inmediatos. |
| Datos estáticos y almacenados (data-at-rest). | Datos que fluyen continuamente (data-in-motion). |
| Entrada finita y conocida. | Entrada infinita, el trabajo nunca termina. |
| Throughput como métrica principal. | Latencia como métrica principal. |

---

## Ciclo de Vida del Dato en Streaming

1. **Generación (Upstream):** Aplicaciones productoras generan el flujo de datos original (sensores, logs).
2. **Distribución (Message Processor):** Sistemas como Kafka organizan y distribuyen los mensajes.
3. **Procesamiento (Stream Processor):** Se aplica lógica de negocio: filtrado, transformación, joins, enriquecimiento.
4. **Consumo/Salida:** Resultados enviados a sistemas externos, alertas o tableros en tiempo real.

---

## ¿Qué es Kafka?

Kafka es una plataforma de mensajería distribuida, diseñada para el procesamiento de flujos de datos en tiempo real.  
Actúa como un broker, recibiendo, almacenando y distribuyendo eventos entre productores y consumidores.  
Ideal para sistemas de alta escala y baja latencia.

---

## Conceptos Clave del Streaming

- **Ventanas temporales (windows):** Agrupan datos en intervalos de tiempo para agregaciones.
- **Watermark:** Maneja datos tardíos, asegurando cierre correcto de ventanas.
- **Triggers:** Definen cuándo se ejecuta la agregación.
- **Estado (state):** Spark puede recordar resultados previos para cálculos acumulados.

---

## Structured Streaming en Spark

Spark Structured Streaming integra DataFrames con procesamiento continuo.  
Permite aplicar transformaciones similares a las de batch, pero en tiempo real.  
Usa ventanas, watermarks y checkpoints para asegurar consistencia.

---

## Ejemplo Práctico: Simulación de Kafka

Simulamos un flujo de eventos usando el source `rate`.  
Transformamos los datos simulados (usuario, producto, monto) y agregamos ventas por minuto.  
Estas métricas alimentan un modelo de Random Forest.  
En producción, solo cambiaríamos la fuente de datos (de rate a Kafka).

---

## Los Tres Pilares de las Aplicaciones de Datos

- **Confiabilidad:** El sistema debe funcionar correctamente ante fallas, garantizando procesamiento "exactamente una vez".
- **Escalabilidad:** Capacidad de manejar crecimiento en volumen o velocidad de datos sin degradar rendimiento.
- **Mantenibilidad:** Fácil de operar y evolucionar para nuevos requisitos de negocio.

---

## Conclusión

El streaming en Spark permite procesar datos en tiempo real, generando, transformando, agregando y modelando datos de manera continua.  
Aunque no tengamos Kafka en la versión free, la lógica es la misma y sienta las bases para un pipeline productivo y escalable.

In [0]:
import time
import json
import random
import os

path = "/Volumes/forecasting/default/data/"

for i in range(20):
    data = [{
        "user_id": random.randint(1, 5),
        "category": random.choice(["A", "B", "C"]),
        "value": random.randint(1, 100)
    }]
    
    file_path = path + f"data_{i}.json"
    
    with open(file_path, "w") as f:
        json.dump(data, f)
    
    print(f"Archivo generado: {file_path}")
    time.sleep(2)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("value", IntegerType(), True)
])

df = spark.readStream \
    .schema(schema) \
    .format("json") \
    .load("/Volumes/forecasting/default/data/")


In [0]:
query = df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/forecasting/default/data") \
    .trigger(availableNow=True) \
    .start()

In [0]:
query
